## **Data Collection (Kaggle = Informasi spam telah diketahui/diberi label)**

In [1]:
import kagglehub
path = kagglehub.dataset_download("tinu10kumar/sms-spam-dataset")

100%|██████████| 2.00M/2.00M [00:00<00:00, 66.5MB/s]

Extracting files...


In [2]:
import pandas as pd
import os
import numpy as np

In [3]:
print(path)
print(os.listdir(path))

/root/.cache/kagglehub/datasets/tinu10kumar/sms-spam-dataset/versions/1
['combined_dataset.csv']


In [4]:
df = pd.read_csv(path+ '/combined_dataset.csv')
df.head()

,target,text
0,spam,Congratulations! You've been selected for a lu...
1,spam,URGENT: Your account has been compromised. Cli...
2,spam,You've won a free iPhone! Claim your prize by ...
3,spam,Act now and receive a 50% discount on all purc...
4,spam,Important notice: Your subscription will expir...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10961 entries, 0 to 10960
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  10961 non-null  object
 1   text    10961 non-null  object
dtypes: object(2)
memory usage: 171.4+ KB


In [6]:
df.columns

Index(['target', 'text'], dtype='object')

In [7]:
df['target'].value_counts(dropna=False)

,count
target,
ham,8555
spam,2406


In [8]:
# encode label
df.columns = ['label','text']
df['label'] = df['label'].map({'ham':0, 'spam':1})

## **Text Preprocessing**

In [9]:
import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    # case folding
    text = text.lower()

    # remove angka & simbol
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # tokenization
    tokens = text.split()

    # stopword removal + stemming
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]

    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


## **Feature Extraction**

TF-IDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

Word Embedding (Word2Vec)

In [11]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.6 MB/s eta 0:00:00


In [12]:
from gensim.models import Word2Vec
import numpy as np

sentences = [text.split() for text in df['clean_text']]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1)

def get_w2v_vector(text):
    words = text.split()
    vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_w2v = np.array([get_w2v_vector(text) for text in df['clean_text']])

## **Modeling (3 Model Wajib)**

In [13]:
from sklearn.model_selection import train_test_split

y = df['label']

# TF-IDF split
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2)

# Word2Vec split
X_train_w2v, X_test_w2v, _, _ = train_test_split(X_w2v, y, test_size=0.2)

Naive Bayes

In [14]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train_tfidf, y_train)

GaussianNB()

Logistic Regression

In [15]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=200)

SVM

In [16]:
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train_tfidf, y_train)

SVC()

## **Model Evaluation & Comparison**

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred)
    }

# TF-IDF Results
print("Naive Bayes:", evaluate(nb, X_test_tfidf, y_test))
print("Logistic Regression:", evaluate(lr, X_test_tfidf, y_test))
print("SVM:", evaluate(svm, X_test_tfidf, y_test))

Naive Bayes: {'Accuracy': 0.875968992248062, 'Precision': 0.6607669616519174, 'Recall': 0.9142857142857143, 'F1-score': 0.7671232876712328}
Logistic Regression: {'Accuracy': 0.9334245326037391, 'Precision': 0.955026455026455, 'Recall': 0.736734693877551, 'F1-score': 0.8317972350230415}
SVM: {'Accuracy': 0.9530323757409941, 'Precision': 0.9685230024213075, 'Recall': 0.8163265306122449, 'F1-score': 0.8859357696566998}


**Interpretasi:**

Naive Bayes
*   Sangat bagus mendeteksi spam (recall tinggi).
*   Tapi cukup sering salah menandai pesan normal sebagai spam (precision lebih rendah).

Logistic Regression
*   Kalau model bilang spam, hampir selalu benar (precision tinggi).
*   Tapi lumayan banyak spam yang lolos (recall lebih rendah).

SVM
*   Ini model dengan hasil terbaik.
*   Akurasi tertinggi.
*   Precision tinggi.
*   Recall juga cukup bagus.
*   F1-score tertinggi (paling seimbang).

**Bandingkan dengan Word2Vec**

In [18]:
lr_w2v = LogisticRegression(max_iter=200)
lr_w2v.fit(X_train_w2v, y_train)

print("LogReg + Word2Vec:", evaluate(lr_w2v, X_test_w2v, y_test))

LogReg + Word2Vec: {'Accuracy': 0.7765617875056999, 'Precision': 0.0, 'Recall': 0.0, 'F1-score': 0.0}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model selalu memprediksi semua data sebagai HAM (bukan spam). Karena dataset spam biasanya imbalanced (spam jauh lebih sedikit), model "malas" dan memilih kelas mayoritas.

Penyebab utama:
* Word2Vec hanya averaging vector
* Dataset imbalanced
* Logistic Regression kurang cocok untuk vector embedding sederhana.

## **Deployment**

In [19]:
import gradio as gr

def predict_spam(text):
    clean = preprocess(text)
    vec = tfidf.transform([clean]).toarray()
    pred = lr.predict(vec)[0]

    return "SPAM 🚨" if pred == 1 else "HAM ✅"

interface = gr.Interface(
    fn=predict_spam,
    inputs="text",
    outputs="text",
    title="Spam Detection App"
)

interface.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://48745bcccaa8095f55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## **Save model**

In [20]:
import joblib
joblib.dump(svm, 'svm_model.pkl')
joblib.dump(tfidf, 'tfidf.pkl')

['tfidf.pkl']